# Step 2 — Testing 4 Classifiers on All 5 Datasets (Within-Project)

In Step 1, we only tested ONE classifier (Logistic Regression) on ONE dataset (CM1),
just to check that everything worked correctly. It did — but the result showed a
problem: because CM1 has very few defective modules (only 10.88%), the classifier
mostly just guessed "not defective" every time.

In this notebook we fix that problem and scale up:

1. We test all 4 classifiers we planned to use: Logistic Regression, Random Forest,
   XGBoost, and a Neural Network (MLP)
2. We test them on all 5 datasets: CM1, JM1, KC1, KC2, PC1
3. We add SMOTE, a technique that fixes the imbalance problem by creating extra
   copies (synthetic examples) of the rare "defective" class during training only

This is still "within-project" — meaning we train and test on the SAME project,
just using cross-validation (splitting the data into 10 parts, testing on each
part in turn). Cross-project testing (training on one project, testing on a
different one) comes in a later notebook.


## Step 2.1 — Import the tools we need

In [ ]:
# pandas and numpy: for handling data tables and numbers
import pandas as pd
import numpy as np
from pathlib import Path

# These come from scikit-learn, the main machine learning library we're using
from sklearn.model_selection import StratifiedKFold   # splits data into folds, keeping class balance similar in each fold
from sklearn.preprocessing import StandardScaler        # rescales features so they're all on a similar scale
from sklearn.linear_model import LogisticRegression     # classifier 1
from sklearn.ensemble import RandomForestClassifier     # classifier 2
from xgboost import XGBClassifier                       # classifier 3
from sklearn.neural_network import MLPClassifier        # classifier 4 (a simple neural network)

# These measure how good our predictions are
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score

# This is the tool that fixes class imbalance by creating synthetic examples
from imblearn.over_sampling import SMOTE

# A fixed "seed" number. This makes random steps (like shuffling data) happen
# the same way every time we run the code, so our results are reproducible.
RANDOM_SEED = 42

## Step 2.2 — Tell the notebook where our cleaned data lives

In [ ]:
# This is the folder where our 5 cleaned CSV files are saved
data_folder = Path(r"C:\Users\kanch\Documents\datascience-thesis\data\processed")

# The names of our 5 datasets
dataset_names = ["cm1", "jm1", "kc1", "kc2", "pc1"]

## Step 2.3 — Load all 5 cleaned datasets

In [ ]:
# We will store each dataset's features (X) and target (y) in two dictionaries.
# A dictionary lets us look things up by name, like all_X["cm1"] to get CM1's features.
all_X = {}
all_y = {}

for name in dataset_names:

    # Read the CSV file for this dataset
    file_path = data_folder / f"{name}_cleaned.csv"
    df = pd.read_csv(file_path)

    # "defects" is the column we are trying to predict (True = defective, False = clean)
    # We convert it to 1s and 0s because that's what scikit-learn expects
    y = df["defects"].astype(int)

    # X is everything EXCEPT the defects column — these are our 21 input features
    X = df.drop(columns=["defects"])

    all_X[name] = X
    all_y[name] = y

    print(f"{name.upper()}: loaded {X.shape[0]} rows and {X.shape[1]} features")

## Step 2.4 — Set up our 4 classifiers

We also need to know WHICH classifiers need feature scaling and which don't.

- Logistic Regression and the Neural Network (MLP) are sensitive to the SIZE of
  numbers — a feature that ranges from 0 to 1000 can unfairly dominate a feature
  that ranges from 0 to 1, even if both are equally important. So we scale these.
- Random Forest and XGBoost work by making yes/no splits on each feature one at a
  time (like a flowchart), so the actual size of the numbers doesn't matter to
  them. We leave these unscaled.


In [ ]:
# A dictionary holding one of each classifier, with some basic settings.
# We are NOT fine-tuning these yet — that comes in a later step. For now we just
# want to see how each one performs with reasonable default settings.
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_SEED),
    "MLP (Neural Network)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_SEED),
}

# A simple list telling us which of the above need their features scaled first
needs_scaling = ["Logistic Regression", "MLP (Neural Network)"]

## Step 2.5 — The main testing loop

This is the most important cell in the notebook, so here is exactly what it does,
step by step, for every (dataset, classifier) combination:

1. Split the dataset into 10 folds (parts), keeping the defect rate similar in each
2. For each fold in turn:
   a. Use 9 folds to TRAIN, and hold back 1 fold to TEST
   b. If this classifier needs scaling, scale the features
      (learning the scaling only from the training fold, never the test fold —
      otherwise we would be "cheating" by letting the model see test data early)
   c. Apply SMOTE to the TRAINING data only, to fix the class imbalance problem
      (we never apply SMOTE to the test data — we want to test on real,
      unmodified examples)
   d. Train the classifier on the balanced training data
   e. Use the trained classifier to predict on the untouched test fold
   f. Measure how good those predictions were (F1, MCC, AUC-ROC)
3. Average the 10 fold results together to get one final score per
   (dataset, classifier) pair

This will take a few minutes to run, since it's training 4 classifiers × 5
datasets × 10 folds = 200 individual training runs in total.


In [ ]:
# This list will collect one row of results for every (dataset, classifier) pair
all_results = []

# Loop over every dataset
for dataset_name in dataset_names:

    X = all_X[dataset_name]
    y = all_y[dataset_name]

    print("=" * 70)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 70)

    # Loop over every classifier
    for classifier_name, classifier in classifiers.items():

        # Split the data into 10 folds. shuffle=True mixes up the row order first,
        # and random_state makes that shuffle repeatable.
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

        # These lists will hold the score from each of the 10 folds
        fold_f1_scores = []
        fold_mcc_scores = []
        fold_auc_scores = []

        # .split() gives us the row numbers to use for training and testing,
        # one fold at a time
        for train_rows, test_rows in skf.split(X, y):

            # Select the actual training and testing data using those row numbers
            X_train = X.iloc[train_rows]
            X_test = X.iloc[test_rows]
            y_train = y.iloc[train_rows]
            y_test = y.iloc[test_rows]

            # Step A: scale the features, but only if this classifier needs it
            if classifier_name in needs_scaling:
                scaler = StandardScaler()
                # "fit_transform" LEARNS the scaling from training data AND applies it
                X_train = scaler.fit_transform(X_train)
                # "transform" only APPLIES the already-learned scaling — it does not
                # learn anything new from the test data
                X_test = scaler.transform(X_test)

            # Step B: apply SMOTE to the training data only, to balance the classes.
            # This creates synthetic "defective" examples so the classifier sees
            # a more balanced mix of defective vs clean during training.
            smote = SMOTE(random_state=RANDOM_SEED)
            X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

            # Step C: train the classifier on the balanced training data
            classifier.fit(X_train_balanced, y_train_balanced)

            # Step D: predict on the untouched, real test data
            y_predicted = classifier.predict(X_test)
            # predict_proba gives a probability instead of just yes/no — needed for AUC-ROC
            y_predicted_probability = classifier.predict_proba(X_test)[:, 1]

            # Step E: measure how good the predictions were, and save the scores
            fold_f1_scores.append(f1_score(y_test, y_predicted, zero_division=0))
            fold_mcc_scores.append(matthews_corrcoef(y_test, y_predicted))
            fold_auc_scores.append(roc_auc_score(y_test, y_predicted_probability))

        # Average the 10 fold scores together for this (dataset, classifier) pair
        average_f1 = np.mean(fold_f1_scores)
        average_mcc = np.mean(fold_mcc_scores)
        average_auc = np.mean(fold_auc_scores)

        print(f"{classifier_name:22s}  F1={average_f1:.3f}   MCC={average_mcc:.3f}   AUC-ROC={average_auc:.3f}")

        # Save this result as one row for our final summary table
        all_results.append({
            "Dataset": dataset_name.upper(),
            "Classifier": classifier_name,
            "F1": round(average_f1, 3),
            "MCC": round(average_mcc, 3),
            "AUC-ROC": round(average_auc, 3),
        })

    print()

## Step 2.6 — View the full results table

In [ ]:
# Turn our list of results into a proper table (a DataFrame) so it's easy to read
results_table = pd.DataFrame(all_results)
results_table

## Step 2.7 — Compare with Step 1 (before SMOTE)

In Step 1, Logistic Regression on CM1 without SMOTE scored:
- F1 = 0.089
- MCC = 0.080
- AUC-ROC = 0.803

Compare that row in the table above (CM1 + Logistic Regression, WITH SMOTE this
time) to see whether SMOTE actually helped. This comparison is worth keeping for
the thesis — it's direct, real evidence of whether the class-imbalance handling
made a practical difference.

## Step 2.8 — Save the results table to a CSV file

In [ ]:
results_folder = Path(r"C:\Users\kanch\Documents\datascience-thesis\experiments\results")
results_folder.mkdir(parents=True, exist_ok=True)

output_path = results_folder / "step2_within_project_results.csv"
results_table.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")